In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("reconciliation_table", "oh_apm_stg.tmp.ctl_reconciliation_log_Prv")
dbutils.widgets.text("error_table", "oh_apm_stg.vendor_extracts.error_load_report_log_cpc_Stg_Prv")
dbutils.widgets.text("s3_path", "s3://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/weekly")

In [0]:
reconciliation_table = dbutils.widgets.get("reconciliation_table")
error_table = dbutils.widgets.get("error_table")
s3_path = dbutils.widgets.get("s3_path")

In [0]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql import Row

# -------------------------------
# Utility function: safe get task values
# -------------------------------
def safe_get(task_key, key, default):
    try:
        return dbutils.jobs.taskValues.get(taskKey=task_key, key=key, debugValue=default)
    except Exception:
        return default

# -------------------------------
# Retrieve pre-validation outputs
# -------------------------------
gz_expected_info = safe_get("Pre_validation", "gz_expected_info", None)
gz_file_paths = safe_get("Pre_validation", "gz_file_paths", None)
missing_required_gz_files = safe_get("Pre_validation", "missing_required_gz_files", None)
ctl_received_ts = safe_get("Pre_validation", "ctl_received_ts", None)
start_load = safe_get("Pre_validation", "start_load", None)

# -------------------------------
# Validate pre-validation output
# -------------------------------
critical_empty = False
if gz_expected_info is None or gz_file_paths is None:
    critical_empty = True
if missing_required_gz_files is None:
    missing_required_gz_files = []

if critical_empty:
    raise Exception("❌ Pre-validation returned empty critical data. Cannot continue Error Load Report.")

# -------------------------------
# Default timestamps if not set
# -------------------------------
if not start_load:
    start_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
end_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
if not ctl_received_ts:
    ctl_received_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# -------------------------------
# Prepare error records
# -------------------------------
error_records = []

# Missing required files take precedence
missing_required_set = set(missing_required_gz_files)
gz_expected_names = [name for name, _ in gz_expected_info]

# Files missing from expected vs actual
missing_gz_files = [name for name in gz_expected_names if name not in gz_file_paths]

# Combine for reporting
all_missing_files = list(missing_required_set) + missing_gz_files

if all_missing_files:
    for file_name, expected_count in gz_expected_info:
        error_records.append((
            file_name,
            "" if file_name in missing_required_set else ctl_received_ts,
            start_load,
            end_load,
            0,
            f"MISSING FILE(S): {', '.join(all_missing_files)}"
        ))

# -------------------------------
# Build error DataFrame
# -------------------------------
error_schema = StructType([
    StructField("File_Name", StringType(), True),
    StructField("Date_Received", StringType(), True),
    StructField("Start_Load_Date", StringType(), True),
    StructField("End_Load_Date", StringType(), True),
    StructField("Row_Number", LongType(), True),
    StructField("Error_Description", StringType(), True)
])

if error_records:
    error_df = spark.createDataFrame(error_records, schema=error_schema)
    # Write to error table if specified
    try:
        error_table = safe_get("Pre_validation", "error_table", None)
        if error_table:
            error_df.write.mode("append").saveAsTable(error_table)
            print(f"✅ Error report written to {error_table}")
            display(error_df)  # Show what was written
        else:
            print("⚠️ No error_table specified. Skipping write.")
        dbutils.jobs.taskValues.set(key="Copy_to_APMTable", value=True)
    except Exception as e:
        raise Exception(f"❌ Failed to write error table: {e}")
else:
    print("✅ No missing files detected.")
    dbutils.jobs.taskValues.set(key="Copy_to_APMTable", value=False)

# -------------------------------
# Optional: print debug info
# -------------------------------
print(f"🔹 Start Load: {start_load}")
print(f"🔹 End Load: {end_load}")
print(f"🔹 Date Received: {ctl_received_ts}")
print(f"🔹 Missing Required Files: {missing_required_gz_files}")
print(f"🔹 Missing GZ Files in S3: {missing_gz_files}")
print(f"🔹 Total Records to Load: {len(error_records)}")
